# 02 — Tasks, budgets, and the ledger via `AgentKernel`

`AgentKernel` is the public Python facade. It wraps the scheduler, the spawn manager, and the JSONL/state stores into a single object. Use it when you want tasks as first-class entities (with budget reservation, refund-on-failure, and policy admission) rather than raw notebook execution.

**Kernel:** standard `python3`.

In [ ]:
import json, tempfile, shutil
from pathlib import Path

import nbformat
from nbformat.v4 import new_notebook, new_code_cell

from agent_kernel.api import AgentKernel

workspace = Path(tempfile.mkdtemp(prefix='ak-ex02-'))
print('workspace:', workspace)

## 1. Open a workspace under the `local-dev` profile

Constructing `AgentKernel(workspace)` opens (or creates) the workspace, loads the requested `PolicyProfile`, and starts the in-process scheduler. The two shipped profiles are `local-dev` (generous defaults) and `research` (tighter caps).

In [ ]:
ak = AgentKernel(workspace, policy_profile='local-dev')
print('profile:', ak.scheduler.profile.name)
print('initial budget:', ak.scheduler.profile.budgets.model_dump())
print('quota:', ak.scheduler._quota_snapshot_locked().model_dump())

## 2. Create + run a task

`create_task` writes the initial `TaskSpec` snapshot and emits `task.created`. `run_task` admits it through the policy engine, executes the notebook via the runner, and emits the full chain through to `task.completed` (or `task.failed`).

In [ ]:
nb_path = workspace / 'work.ipynb'
nbformat.write(new_notebook(
    cells=[
        new_code_cell('total = sum(range(10))'),
        new_code_cell('total'),
    ],
    metadata={'kernelspec': {'name': 'python3', 'display_name': 'Python 3'}},
), nb_path)

task = ak.create_task(notebook_path=str(nb_path), kernel_name='python3')
print('created:  ', task.task_id, task.status.value)
print('reserved: ', task.reserved_budget.model_dump())

final = ak.run_task(task.task_id)
print('final:    ', final.status.value)
print('spent:    ', final.spent_budget.model_dump())
print('executed: ', final.executed_notebook_path)

## 3. Inspect the event chain for this task

`list_events(task_id)` filters the JSONL ledger to one task — useful for test assertions and for live introspection.

In [ ]:
for e in ak.list_events(task.task_id):
    print(f'{e.ts}  {e.event_type.value:35s}  {(e.payload or {}).get("reason", "")}')

## 4. Show the on-disk state snapshot

The scheduler atomically writes a snapshot of every task to `<ws>/.agent_kernel/tasks/<task_id>.json` (temp-write-then-rename). The same object is recoverable from the JSONL alone — see `scripts/replay.py`.

In [ ]:
snapshot = workspace / '.agent_kernel' / 'tasks' / f'{task.task_id}.json'
print(snapshot.read_text())

## 5. Run a task that fails — see refund on the budget

A failed task **refunds its unspent reservation** so the running budget stays exact. We'll write a notebook that raises in cell 1 and watch for `task.failed` plus the corresponding `budget.refunded`.

In [ ]:
fail_nb = workspace / 'boom.ipynb'
nbformat.write(new_notebook(
    cells=[new_code_cell('raise RuntimeError("boom")')],
    metadata={'kernelspec': {'name': 'python3', 'display_name': 'Python 3'}},
), fail_nb)

failing = ak.create_task(notebook_path=str(fail_nb), kernel_name='python3')
out = ak.run_task(failing.task_id)
print('final status:', out.status.value)
print()
events = ak.list_events(failing.task_id)
for e in events:
    print(f'{e.event_type.value:35s}  {(e.payload or {}).get("reason", "")}')

In [ ]:
shutil.rmtree(workspace, ignore_errors=True)